# Chapter 4 self-supervised architecture comparison

**Status:** canonical thesis experiment  
**Thesis sections:** 2.4, 3, 4

This notebook shows the key interface that made self-supervision possible: PyTorch proposes cellwise SUPG parameters, DOLFINx evaluates the state/adjoint objective and its gradient, and PyTorch receives that gradient during `backward()`. The experiment loop is configuration driven so all five architecture runs remain comparable.

## The differentiable FEM bridge

For weights $y$, `forward` transfers a detached array to the FEM solver and returns $J(y)$. The adjoint calculation supplies $\nabla_yJ$; `backward` transfers it back to the tensor device. This avoids differentiating through a sparse linear solve with PyTorch while retaining an exact discrete-adjoint gradient.

In [ ]:
import torch
from supgml.autograd import FEMObjective

# `solver` is the AdjointSUPGSolver created in notebook 01, or loaded with a case.
# fem_objective = FEMObjective(solver)
# predicted_tau = torch.nn.Parameter(torch.as_tensor(solver.yh.x.array).reshape(-1, 1))
# loss = fem_objective(predicted_tau)
# loss.backward()
# assert predicted_tau.grad.shape == predicted_tau.shape


The custom `torch.autograd.Function` lives in `supgml.autograd` because it is independent of a particular PDE. State equations, objectives, and mesh data remain explicit in the data-generation notebooks.

In [ ]:
from supgml.experiments import load_config, project_root

config = load_config(project_root() / "experiments/ch4_self_supervised.json")
config


Run from the repository root with `supgml-train experiments/ch4_self_supervised.json` inside the supported DOLFINx environment. The batch size remains one because each graph invokes its corresponding state-and-adjoint FEM solver. The command stores checkpoints and loss histories under `runs/ch4_self_supervised`.